# SQL 8: window functions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-08-window-functions.ipynb)

A **window function** computes a value over a set of rows related to the current row, **without collapsing them**.
A `GROUP BY` returns one row per group. A window keeps every row and adds a column, so you can show each order next
to its customer's running total, or next to its rank.

The syntax is `function() OVER (PARTITION BY ... ORDER BY ... frame)`. `PARTITION BY` restarts the window for each
group, `ORDER BY` sets the sequence, and the frame chooses which rows are included. This chapter builds up those
parts one at a time and ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW pay AS
SELECT * FROM VALUES ('Alice', 150), ('Bob', 120), ('Carol', 120), ('Dave', 95)
AS pay(name, salary)
""")

The examples use three small tables. `orders` has 14 orders from six customers, with three statuses
(`completed`, `cancelled`, `pending`). Customer 1 has three completed orders of 100, 200 and 150, which is small
enough to check every result by hand. `pay` has four salaries, and Bob and Carol tie at 120, so it shows how
each function treats ties.

In [ ]:
assert spark.sql("SELECT COUNT(*) FROM orders").first()[0] == 14
assert spark.sql("SELECT COUNT(DISTINCT customer_id) FROM orders").first()[0] == 6
spark.sql("SELECT * FROM pay ORDER BY salary DESC, name").show()

## 1. A window keeps every row

Some questions need a value computed over a group of rows, shown next to each row of that group. "Show each
order with its customer's total spend" is one. A `GROUP BY` cannot answer it in one step, because it collapses
each group into a single row and the individual orders are gone.

A **window function** solves this. For every row, called the **current row**, it looks at a set of related rows,
called the **window**, computes one value from them, and attaches that value to the current row. No rows are
removed. The result has as many rows as the input, plus one new column.

The window is described inside `OVER (...)`, which has three optional parts:

| Part | Question it answers | Example |
|---|---|---|
| `PARTITION BY` | Which rows belong to the same group as the current row? | `PARTITION BY customer_id` |
| `ORDER BY` | In what sequence are the rows of a group read? | `ORDER BY order_date` |
| frame | Which rows of the group, relative to the current row, does the function use? | `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` |

A **partition** is the set of rows that share the same `PARTITION BY` values. Each partition is processed on its
own, as if it were a separate table. An empty `OVER ()` makes the whole result a single partition.

### Where you'll use it

- **Analytics without losing detail:** showing each order next to its customer's total, which a `GROUP BY` cannot do in one step.
- **Data-quality checks:** `COUNT(*) OVER (PARTITION BY key)` flags keys that appear more than once, while keeping the rows so you can inspect them.

### Worked example: GROUP BY against a window

Customer 1 has three orders: 100, 200 and 150. Their total is 450.

- `SUM(amount) ... GROUP BY customer_id` returns one row for customer 1, with 450.
- `SUM(amount) OVER (PARTITION BY customer_id)` returns all three order rows, and each carries 450.

The code below runs both. The grouped query has 6 rows, one per customer. The window query has 14 rows, one per
order. `COUNT(*) OVER ()` has an empty window, so it counts the whole result and gives 14 on every row.

In [ ]:
grouped = spark.sql("""
SELECT customer_id, SUM(amount) AS customer_total
FROM orders
GROUP BY customer_id
ORDER BY customer_id
""")
grouped.show(3)

windowed = spark.sql("""
SELECT order_id, customer_id, amount,
       SUM(amount) OVER (PARTITION BY customer_id) AS customer_total,
       COUNT(*)    OVER ()                         AS all_orders
FROM orders
ORDER BY customer_id, order_id
""")
windowed.show(4)

assert grouped.count() == 6
assert windowed.count() == 14
customer_1 = [(r.order_id, r.amount, r.customer_total) for r in windowed.collect() if r.customer_id == 1]
assert customer_1 == [(1, 100, 450), (2, 200, 450), (3, 150, 450)]
assert {r.all_orders for r in windowed.collect()} == {14}

### When the window is computed

SQL clauses run in a fixed logical order, which is not the order they are written in:

1. `FROM` and joins build the input rows.
2. `WHERE` removes rows.
3. `GROUP BY` collapses rows into groups, and `HAVING` removes groups.
4. **Window functions** are computed.
5. The `SELECT` list is evaluated, then `DISTINCT`.
6. `ORDER BY` sorts, and `LIMIT` cuts.

Two consequences follow, and the rest of the chapter relies on them. First, a window only sees the rows that
survived `WHERE` and `GROUP BY`, so filtering before a window changes its result. Second, a window result does not
exist yet when `WHERE` runs, so `WHERE` cannot filter on it (section 7 shows the fix).

Here the same window runs with and without `WHERE status = 'completed'`. Customer 2 has orders of 300, 250
(cancelled) and 400. With the filter, the cancelled order is gone before the window runs, so the total drops from
950 to 700.

In [ ]:
q = """
SELECT 'all' AS rows_seen, SUM(amount) OVER (PARTITION BY customer_id) AS customer_total
FROM orders WHERE customer_id = 2
UNION ALL
SELECT 'completed', SUM(amount) OVER (PARTITION BY customer_id)
FROM orders WHERE customer_id = 2 AND status = 'completed'
"""
spark.sql(q).show()
assert {(r.rows_seen, r.customer_total) for r in spark.sql(q).collect()} == {("all", 950), ("completed", 700)}

## 2. Ranking: ROW_NUMBER, RANK and DENSE_RANK

A **ranking function** numbers the rows of a window in the order given by `ORDER BY`. It needs no argument and no
frame: the position of the row in the sorted window is all it uses. Rows that have the same `ORDER BY` value are
called **ties** (or **peers**). The three ranking functions differ only in how they number ties.

- `ROW_NUMBER()` gives every row a different number: 1, 2, 3, ... Tied rows still get different numbers, and which
  tied row comes first is not defined. Spark may pick either one, and the choice can change between runs. Add a
  **tie-breaker**, an extra `ORDER BY` column that is unique, such as `name` or an id, to make the result
  **deterministic** (the same on every run).
- `RANK()` gives tied rows the same number. The next row's number is 1 plus the count of rows ahead of it, so the
  sequence skips: 1, 2, 2, 4.
- `DENSE_RANK()` gives tied rows the same number, and the next distinct value gets the next integer, so there is
  no gap: 1, 2, 2, 3.

### Where you'll use it

- **Deduplication:** `ROW_NUMBER() OVER (PARTITION BY key ORDER BY updated_at DESC)` keeps the latest version of each record; [SQL 9: deduplication](sql-09-deduplication.ipynb) covers the
  full pattern.
- **Leaderboards:** top sellers per region with `RANK` or `DENSE_RANK`, so tied sellers share a place.
- **"Nth highest" questions:** `DENSE_RANK() = 2` finds the second-highest distinct value, even when the highest value repeats. [SQL 13](sql-13-interview-drills.ipynb), Part A, drills these.

### Worked example

Sorted by salary, highest first, Bob and Carol tie at 120:

| name | salary | ROW_NUMBER | RANK | DENSE_RANK |
|---|---|---|---|---|
| Alice | 150 | 1 | 1 | 1 |
| Bob | 120 | 2 | 2 | 2 |
| Carol | 120 | 3 | 2 | 2 |
| Dave | 95 | 4 | 4 | 3 |

Dave's `RANK` is 4 because three people earn more than him. His `DENSE_RANK` is 3 because only two distinct
salaries (150 and 120) are higher. `ROW_NUMBER` uses `name` as the tie-breaker, so Bob is 2 and Carol is 3.

In [ ]:
q = """
SELECT name, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC, name) AS row_number,
       RANK()       OVER (ORDER BY salary DESC)       AS rank,
       DENSE_RANK() OVER (ORDER BY salary DESC)       AS dense_rank
FROM pay
ORDER BY salary DESC, name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("Alice", 150, 1, 1, 1),
    ("Bob", 120, 2, 2, 2),
    ("Carol", 120, 3, 2, 2),
    ("Dave", 95, 4, 4, 3),
]

Without the tie-breaker, `ROW_NUMBER` still gives Bob and Carol the numbers 2 and 3, but the SQL does not say
which of them gets 2. The only safe assertion is about the pair, not about either person.

In [ ]:
q = """
SELECT name, ROW_NUMBER() OVER (ORDER BY salary DESC) AS rn
FROM pay
"""
rows = {r.rn: r.name for r in spark.sql(q).collect()}
assert rows[1] == "Alice" and rows[4] == "Dave"
assert {rows[2], rows[3]} == {"Bob", "Carol"}  # either order is allowed

## 3. Running totals with PARTITION BY and ORDER BY

A **running total** (or cumulative sum) is, for each row, the sum of that row and every row before it. Adding
`ORDER BY` inside `OVER` turns an aggregate such as `SUM` into a running aggregate: for each row, the function
reads the partition from its first row up to the current row, in the given order. Without `ORDER BY`, the same
`SUM` reads the whole partition, which is the total you saw in section 1.

`PARTITION BY customer_id` restarts the running total for each customer. The `order_id` after `order_date` is a
tie-breaker: if a customer placed two orders on the same date, the order would otherwise be undefined, and the
default frame would give both orders the same total (section 4 explains why).

### Where you'll use it

- **Cumulative revenue** per customer or per month, and running balances in ledgers.
- **Lifetime value to date:** how much a customer had spent at the moment of each order, a common feature in churn and fraud models.

### Worked example

Completed orders only (the `WHERE` runs first, so customer 2's cancelled order is not counted):

| customer | order_date | amount | running_total | customer_total |
|---|---|---|---|---|
| 1 | 2023-01-05 | 100 | 100 | 450 |
| 1 | 2023-02-10 | 200 | 100 + 200 = 300 | 450 |
| 1 | 2023-03-15 | 150 | 300 + 150 = 450 | 450 |
| 2 | 2023-01-20 | 300 | 300 (restarts) | 700 |
| 2 | 2023-03-20 | 400 | 300 + 400 = 700 | 700 |

The last running total of each partition equals the partition total.

In [ ]:
q = """
SELECT customer_id, order_id, amount,
       SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS running_total,
       SUM(amount) OVER (PARTITION BY customer_id)                               AS customer_total
FROM orders
WHERE status = 'completed' AND customer_id IN (1, 2)
ORDER BY customer_id, order_date
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, 1, 100, 100, 450),
    (1, 2, 200, 300, 450),
    (1, 3, 150, 450, 450),
    (2, 4, 300, 300, 700),
    (2, 6, 400, 700, 700),
]

## 4. Frames: ROWS versus RANGE

The **frame** is the part of the partition that an aggregate reads for the current row. It is written as
`ROWS BETWEEN start AND end` or `RANGE BETWEEN start AND end`, where each bound is one of:

- `UNBOUNDED PRECEDING`: the first row of the partition.
- `n PRECEDING`: n before the current row.
- `CURRENT ROW`.
- `n FOLLOWING`: n after the current row.
- `UNBOUNDED FOLLOWING`: the last row of the partition.

`ROWS` and `RANGE` differ in what "before" and "current" mean:

- `ROWS` counts **physical rows** in the sorted order. `2 PRECEDING` means the two rows above, and `CURRENT ROW`
  means this row alone.
- `RANGE` works on the **values** of the `ORDER BY` column. `CURRENT ROW` means the current row and all its peers
  (rows with the same value), and `25 PRECEDING` means "rows whose value is at most 25 below mine".

When the window has `ORDER BY` but no frame, the **default frame** is
`RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. Without `ORDER BY`, the default frame is the whole partition.
The difference only shows when the `ORDER BY` value has ties, which is why you should write the frame explicitly
whenever ties can occur.

### Where you'll use it

- **Moving averages:** a 7-day rolling average of daily sales (`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` over one row per day), the standard smoothing for noisy metrics.
- **Correct running totals:** making sure two events with the same timestamp do not get the same cumulative value by accident.

### Worked example

Sorted by salary, lowest first: Dave 95, Bob 120, Carol 120, Alice 150. Bob and Carol are peers.

| name | salary | ROWS running total | RANGE running total |
|---|---|---|---|
| Dave | 95 | 95 | 95 |
| Bob | 120 | 95 + 120 = 215 | 95 + 120 + 120 = 335 |
| Carol | 120 | 215 + 120 = 335 | 335 (same peers as Bob) |
| Alice | 150 | 485 | 485 |

For Bob, `ROWS` stops at Bob's own row. `RANGE` includes every row whose salary is at most 120, which includes
Carol, so Bob and Carol share 335. The `default_total` column has no frame clause and matches `RANGE`.

In [ ]:
q = """
SELECT name, salary,
       SUM(salary) OVER (ORDER BY salary, name
                         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)  AS rows_total,
       SUM(salary) OVER (ORDER BY salary
                         RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS range_total,
       SUM(salary) OVER (ORDER BY salary)                                   AS default_total
FROM pay
ORDER BY salary, name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("Dave", 95, 95, 95, 95),
    ("Bob", 120, 215, 335, 335),
    ("Carol", 120, 335, 335, 335),
    ("Alice", 150, 485, 485, 485),
]

### RANGE with a numeric offset

Because `RANGE` measures distance in values, `RANGE BETWEEN 25 PRECEDING AND 25 FOLLOWING` means "every row whose
salary is within 25 of mine". Dave (95) sees salaries from 70 to 120: himself, Bob and Carol, so 3 rows. Alice (150)
sees 125 to 175: only herself. A `ROWS` frame of the same size would count positions instead, and give different
answers.

In [ ]:
q = """
SELECT name, salary,
       COUNT(*) OVER (ORDER BY salary RANGE BETWEEN 25 PRECEDING AND 25 FOLLOWING) AS within_25,
       COUNT(*) OVER (ORDER BY salary, name ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING) AS neighbours_and_self
FROM pay
ORDER BY salary, name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("Dave", 95, 3, 2),
    ("Bob", 120, 3, 3),
    ("Carol", 120, 3, 3),
    ("Alice", 150, 1, 2),
]

## 5. LAG: the previous row

`LAG(x)` returns the value of `x` from the row before the current row, in the window's `ORDER BY` order. It
ignores the frame. The full form is `LAG(x, offset, default)`:

- `offset` is how many rows back to look (1 if omitted).
- `default` is returned when there is no such row, which happens for the first row of every partition. It is
  `NULL` if omitted.

"The previous row" is only meaningful at the right **grain**, meaning what one row represents. To compare each
month with the month before, first aggregate to one row per month, then apply `LAG`. Applied to raw orders, `LAG`
returns the previous *order*, which is usually in the same month.

### Where you'll use it

- **Change over time:** month-over-month growth, or the time since a customer's last order.
- **Sessionisation:** comparing each event with the previous one to detect a gap that starts a new session.
- **Change detection:** finding the rows where a status differs from the previous row's status.

### Worked example: month-over-month growth

Completed revenue per month: January is 100 + 300 + 120 + 500 = 1020, February is 200 + 180 + 275 = 655, and March
is 150 + 400 + 320 + 450 = 1320. Growth is `(revenue - previous) / previous * 100`:

| month | revenue | previous_revenue | growth_pct |
|---|---|---|---|
| 2023-01 | 1020 | NULL | NULL |
| 2023-02 | 655 | 1020 | (655 - 1020) / 1020 * 100 = -35.78 |
| 2023-03 | 1320 | 655 | (1320 - 655) / 655 * 100 = 101.53 |

January has no previous month, so `LAG` returns `NULL`, and arithmetic with `NULL` gives `NULL`. The query
computes `LAG` once in a CTE and reuses it, rather than repeating the window expression three times. Sorting by the
`'yyyy-MM'` string works because the month is zero-padded, so text order equals date order.

In [ ]:
q = """
WITH monthly AS (
  SELECT date_format(order_date, 'yyyy-MM') AS month, SUM(amount) AS revenue
  FROM orders
  WHERE status = 'completed'
  GROUP BY date_format(order_date, 'yyyy-MM')
),
with_previous AS (
  SELECT month, revenue, LAG(revenue) OVER (ORDER BY month) AS previous_revenue
  FROM monthly
)
SELECT month, revenue, previous_revenue,
       ROUND(100.0 * (revenue - previous_revenue) / previous_revenue, 2) AS growth_pct
FROM with_previous
ORDER BY month
"""
spark.sql(q).show()
rows = spark.sql(q).collect()
assert [(r.month, r.revenue, r.previous_revenue) for r in rows] == [
    ("2023-01", 1020, None),
    ("2023-02", 655, 1020),
    ("2023-03", 1320, 655),
]
assert rows[0].growth_pct is None
assert [float(r.growth_pct) for r in rows[1:]] == [-35.78, 101.53]

### The grain matters

The same `LAG` over raw completed orders, ordered by date, compares each order with the order placed just before
it. The first two orders of 2023 are on 2 January (500) and 5 January (100), so the "previous" value for the
5 January order is another January order, not December's revenue. The `default` argument is used here too: the
first row gets 0 instead of `NULL`.

In [ ]:
q = """
SELECT order_date, amount,
       LAG(amount, 1, 0) OVER (ORDER BY order_date, order_id) AS previous_order_amount
FROM orders
WHERE status = 'completed'
ORDER BY order_date, order_id
LIMIT 3
"""
spark.sql(q).show()
assert [(str(r.order_date), r.amount, r.previous_order_amount) for r in spark.sql(q).collect()] == [
    ("2023-01-02", 500, 0),
    ("2023-01-05", 100, 500),
    ("2023-01-20", 300, 100),
]

## 6. Share of total

A **share of total** divides each row's value by the sum over a larger group, such as the whole result. A window
computes the denominator on every row, so no second query or join is needed.

`SUM(SUM(amount)) OVER ()` nests a window around an aggregate, which looks odd until you recall the logical order
from section 1. `GROUP BY` runs first and produces one row per customer, with `SUM(amount)` as that customer's
total. The window runs next, over those six grouped rows. Its empty `OVER ()` makes all six rows one partition, so
the outer `SUM` adds up the six customer totals.

### Where you'll use it

- **Mix analysis:** each product's share of category revenue, computed with `SUM(...) OVER (PARTITION BY category)`.
- **Pareto checks:** what share of revenue the top 20% of customers bring in, using a running share over customers sorted by spend.

### Worked example

Completed totals per customer are 450, 700, 300, 320, 950 and 275, which add up to 2995. Customer 5's share is
950 / 2995 = 31.72%. Rounded shares can add up to slightly more or less than 100, so the check allows a small
tolerance.

In [ ]:
q = """
SELECT customer_id,
       SUM(amount) AS total,
       SUM(SUM(amount)) OVER () AS grand_total,
       ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct_of_total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show()
rows = spark.sql(q).collect()
assert [r.total for r in rows] == [450, 700, 300, 320, 950, 275]
assert {r.grand_total for r in rows} == {2995}
assert float(rows[4].pct_of_total) == 31.72
assert abs(sum(float(r.pct_of_total) for r in rows) - 100) < 0.05

Adding `PARTITION BY` changes the denominator from the grand total to a group total. Here each of customer 1's
orders is divided by customer 1's total of 450: 100 / 450 = 22.22%, 200 / 450 = 44.44% and 150 / 450 = 33.33%.

In [ ]:
q = """
SELECT order_id, amount,
       ROUND(100.0 * amount / SUM(amount) OVER (PARTITION BY customer_id), 2) AS pct_of_customer
FROM orders
WHERE customer_id = 1
ORDER BY order_id
"""
spark.sql(q).show()
assert [(r.order_id, float(r.pct_of_customer)) for r in spark.sql(q).collect()] == [
    (1, 22.22),
    (2, 44.44),
    (3, 33.33),
]

## 7. Filtering on a window result

A window cannot appear in `WHERE`. Section 1 showed why: `WHERE` runs before window functions, so the window value
does not exist yet when `WHERE` is evaluated. For the same reason it cannot appear in `HAVING` or `GROUP BY`.

The fix is to compute the window in an inner query (a subquery or a CTE) and filter its output in an outer query.
Some warehouses (Snowflake, BigQuery, Databricks) offer a `QUALIFY` clause that filters on a window directly, but
Spark SQL 4.0 does not, so the subquery form is the portable one. The first cell shows Spark rejecting the
direct form.

### Where you'll use it

- **Latest record per key:** a window function in a subquery, filtered to `rn = 1`, is the standard pattern in dbt and warehouse models ([SQL 9](sql-09-deduplication.ipynb)).
- **Top N per group:** the three best-selling products in each category, with `rn <= 3`.

In [ ]:
from pyspark.errors import AnalysisException

try:
    spark.sql("""
    SELECT customer_id, order_id, amount
    FROM orders
    WHERE ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) = 1
    """).collect()
    rejected = False
except AnalysisException:
    rejected = True
assert rejected, "a window function in WHERE should be rejected"

### Worked example: the largest order per customer

The inner query numbers each customer's orders from the largest amount down, with `order_id` as the tie-breaker.
The outer query keeps `rn = 1`. Customer 6's largest order is the cancelled 325, because this query counts every
status. Use `RANK() ... = 1` instead of `ROW_NUMBER` if tied orders should all be returned.

In [ ]:
q = """
SELECT customer_id, order_id, amount
FROM (
  SELECT customer_id, order_id, amount,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn
  FROM orders
)
WHERE rn = 1
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 2, 200), (2, 6, 400), (3, 8, 180), (4, 10, 320), (5, 11, 500), (6, 14, 325)]

## More window functions

The functions below follow the same rules as the earlier sections: the partition and order come from `OVER`, and
aggregate-style functions read the frame. Each subsection has one example.

### Where you'll use it

- **Quartiles for segmentation:** `NTILE(4)` splits customers into spend quartiles.
- **Fill-forward and first-value logic:** `FIRST_VALUE` and `LAST_VALUE` carry the first or latest known status onto other rows.
- **Next-event logic:** `LEAD` finds when a subscription was replaced by the next one, to compute how long each lasted.

### LEAD: the next row

`LEAD(x, offset, default)` is the mirror of `LAG`: it reads the row after the current one. The last row of each
partition has no next row and gets `NULL`. Customer 1's orders are 100, 200, 150 in date order, so the next amounts
are 200, 150 and `NULL`.

In [ ]:
q = """
SELECT customer_id, order_date, amount,
       LEAD(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS next_amount
FROM orders
WHERE customer_id IN (1, 2)
ORDER BY customer_id, order_date
"""
spark.sql(q).show()
assert [(r.customer_id, r.amount, r.next_amount) for r in spark.sql(q).collect()] == [
    (1, 100, 200),
    (1, 200, 150),
    (1, 150, None),
    (2, 300, 250),
    (2, 250, 400),
    (2, 400, None),
]

### Moving average

A **moving average** averages a fixed number of recent rows, so the window slides forward one row at a time. The
frame `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` holds the current order and up to two before it.

At the start of a partition there are fewer rows to average. The first order averages only itself (100), and the
second averages two orders ((100 + 200) / 2 = 150). Only from the third row on is the frame full: (100 + 200 + 150)
/ 3 = 150. The `rows_in_frame` column makes this visible. A report that must only show full windows can filter on it.

In [ ]:
q = """
SELECT customer_id, order_date, amount,
       ROUND(AVG(amount) OVER (PARTITION BY customer_id ORDER BY order_date
                               ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS moving_avg_3,
       COUNT(*) OVER (PARTITION BY customer_id ORDER BY order_date
                      ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS rows_in_frame
FROM orders
WHERE customer_id = 1
ORDER BY order_date
"""
spark.sql(q).show()
assert [(r.amount, r.moving_avg_3, r.rows_in_frame) for r in spark.sql(q).collect()] == [
    (100, 100.0, 1),
    (200, 150.0, 2),
    (150, 150.0, 3),
]

### FIRST_VALUE and LAST_VALUE

`FIRST_VALUE(x)` and `LAST_VALUE(x)` return `x` from the first and last row of the **frame**, not of the partition.
That distinction matters because of the default frame from section 4: with `ORDER BY` and no frame, the frame ends
at the current row, so `LAST_VALUE` returns the current row's own value. To get the last value of the whole
partition, spell out `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.

Customer 2's orders are 300, 250 and 400. `FIRST_VALUE` is 300 on every row, because every frame starts at the
first row. `last_default` repeats each row's own amount, and `last_amount` with the full frame is 400 on every row.

In [ ]:
q = """
SELECT customer_id, order_date, amount,
       FIRST_VALUE(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS first_amount,
       LAST_VALUE(amount)  OVER (PARTITION BY customer_id ORDER BY order_date) AS last_default,
       LAST_VALUE(amount)  OVER (PARTITION BY customer_id ORDER BY order_date
                                 ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_amount
FROM orders
WHERE customer_id = 2
ORDER BY order_date
"""
spark.sql(q).show()
assert [(r.amount, r.first_amount, r.last_default, r.last_amount) for r in spark.sql(q).collect()] == [
    (300, 300, 300, 400),
    (250, 300, 250, 400),
    (400, 300, 400, 400),
]

### NTILE: equal-sized buckets

`NTILE(n)` sorts the rows and deals them into `n` numbered buckets of nearly equal size. When the row count does
not divide evenly, the first buckets get one extra row each. With 14 orders and 4 buckets, 14 = 4 x 3 + 2, so the
first two buckets get 4 rows and the last two get 3. Bucket 1 holds the smallest amounts.

In [ ]:
q = """
SELECT order_id, amount, NTILE(4) OVER (ORDER BY amount, order_id) AS quartile
FROM orders
ORDER BY amount, order_id
"""
spark.sql(q).show()
rows = spark.sql(q).collect()
sizes = [sum(1 for r in rows if r.quartile == bucket) for bucket in (1, 2, 3, 4)]
assert sizes == [4, 4, 3, 3]
assert [r.amount for r in rows if r.quartile == 1] == [100, 120, 150, 180]

### PERCENT_RANK and CUME_DIST

Two functions express a rank as a fraction between 0 and 1, which is easier to compare across groups of different
sizes. With `n` rows in the partition:

- `PERCENT_RANK()` = (`RANK()` - 1) / (n - 1). The first row is 0 and the last is 1.
- `CUME_DIST()` = (number of rows sorted at or before the current row, peers included) / n.

For `pay` sorted by salary, highest first: Alice has rank 1, so percent rank 0 and cume dist 1/4 = 0.25. Bob and
Carol have rank 2, so percent rank 1/3, and three of the four rows are at or before them, so cume dist 0.75. Dave
has rank 4, so percent rank 3/3 = 1.0 and cume dist 1.0.

In [ ]:
q = """
SELECT name, salary,
       ROUND(PERCENT_RANK() OVER (ORDER BY salary DESC), 4) AS percent_rank,
       CUME_DIST()          OVER (ORDER BY salary DESC)     AS cume_dist
FROM pay
ORDER BY salary DESC, name
"""
spark.sql(q).show()
assert [(r.name, r.percent_rank, r.cume_dist) for r in spark.sql(q).collect()] == [
    ("Alice", 0.0, 0.25),
    ("Bob", 0.3333, 0.75),
    ("Carol", 0.3333, 0.75),
    ("Dave", 1.0, 1.0),
]

### Named windows: the WINDOW clause

When several functions use the same window, a `WINDOW` clause names it once (`WINDOW w AS (...)`) and each function
refers to it with `OVER w`. The query is shorter, and the functions cannot drift apart when one of them is edited.
The clause comes after `WHERE`, `GROUP BY` and `HAVING`, and before `ORDER BY`. As with any window, rows removed by
`WHERE` are not seen.

In [ ]:
q = """
SELECT customer_id, order_id,
       ROW_NUMBER() OVER w AS nth_order,
       SUM(amount)  OVER w AS running_total
FROM orders
WHERE customer_id = 1
WINDOW w AS (PARTITION BY customer_id ORDER BY order_date, order_id)
ORDER BY customer_id, nth_order
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 1, 1, 100), (1, 2, 2, 300), (1, 3, 3, 450)]

## Interview notes

- **Window versus GROUP BY.** A window keeps every input row and adds a column; `GROUP BY` returns one row per
  group. Use a window when you need the detail row and the group value side by side.
- **The default frame.** With `ORDER BY` and no frame, the frame is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT
  ROW`. Rows that tie on the order value share one running total. Add `ROWS` when each row should count alone.
  Without `ORDER BY`, the frame is the whole partition.
- **LAST_VALUE trap.** Because of that default frame, `LAST_VALUE` returns the current row unless you extend the
  frame to `UNBOUNDED FOLLOWING`.
- **ROW_NUMBER vs RANK vs DENSE_RANK.** Use `ROW_NUMBER` for exactly one row per group, with a unique tie-breaker so
  the result is deterministic. Use `RANK` or `DENSE_RANK` when ties should all appear; `DENSE_RANK` for "Nth
  highest distinct value".
- **A window cannot appear in WHERE.** Windows are computed after `WHERE`, `GROUP BY` and `HAVING`. Compute the
  window in a subquery or CTE, then filter. `QUALIFY` exists in some warehouses but not in Spark SQL 4.0.
- **LAG and LEAD at the edges.** The first (or last) row of each partition gets `NULL`, or the third argument if
  you give one.
- **Cost.** In Spark, a window shuffles rows so that each partition is on one task, then sorts it. A window with no
  `PARTITION BY` sends every row to a single task, which does not scale. Partition the window whenever the logic
  allows.

## Summary

- A window function computes a value over related rows and keeps every row. A `GROUP BY` collapses them.
- `PARTITION BY` sets the groups, `ORDER BY` sets the sequence, and the frame sets which rows an aggregate reads.
- Windows run after `WHERE`, `GROUP BY` and `HAVING`, and before `ORDER BY`. Filter on a window result in an
  outer query.
- Give `ROW_NUMBER` a unique tie-breaker, and make the frame explicit when ties can occur.
- `LAG` and `LEAD` look at neighbouring rows. Aggregate to the right grain first.

## Practice questions

1. Rank the customers by their total completed amount, highest first. Show `customer_id`, the total and the rank,
   sorted by rank.
2. For customer 1, show each order with the amount of the previous order. Sort by `order_id`. The first order has
   no previous order.
3. For each customer, show the amount of their second-largest order, counting every status. Sort by `customer_id`.
4. Show completed revenue per month and a running total across months, sorted by month.
5. For customer 1, show each order's date and the number of days since their previous order. Sort by `order_id`.

## Answers

### Question 1

Aggregate first, so there is one row per customer, then rank those rows. No customer totals tie, so `RANK`,
`DENSE_RANK` and `ROW_NUMBER` would all give the same numbers here.

In [ ]:
q = """
WITH totals AS (
  SELECT customer_id, SUM(amount) AS total
  FROM orders
  WHERE status = 'completed'
  GROUP BY customer_id
)
SELECT customer_id, total, RANK() OVER (ORDER BY total DESC) AS rank
FROM totals
ORDER BY rank
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(5, 950, 1), (2, 700, 2), (1, 450, 3), (4, 320, 4), (3, 300, 5), (6, 275, 6)]

### Question 2

`LAG` ordered by date reads the previous order. The window's order and the output's `ORDER BY` are separate, so the
result can be sorted by `order_id` while `LAG` follows `order_date`.

In [ ]:
q = """
SELECT order_id, amount,
       LAG(amount) OVER (ORDER BY order_date) AS previous_amount
FROM orders
WHERE customer_id = 1
ORDER BY order_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 100, None), (2, 200, 100), (3, 150, 200)]

### Question 3

Number each customer's orders from the largest down and keep number 2. If a customer could have two orders with the
same largest amount, decide what "second-largest" means: `ROW_NUMBER` would return the tied duplicate, while
`DENSE_RANK() = 2` returns the next smaller distinct amount. Here no customer has tied amounts, and `order_id`
keeps the numbering deterministic.

In [ ]:
q = """
SELECT customer_id, amount AS second_largest
FROM (
  SELECT customer_id, amount,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn
  FROM orders
)
WHERE rn = 2
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 150), (2, 300), (3, 120), (4, 220), (5, 450), (6, 275)]

### Question 4

Aggregate to one row per month, then apply a running `SUM` ordered by month. Months are unique after the
`GROUP BY`, so `ROWS` and `RANGE` give the same result: 1020, 1020 + 655 = 1675, and 1675 + 1320 = 2995.

In [ ]:
q = """
WITH monthly AS (
  SELECT date_format(order_date, 'yyyy-MM') AS month, SUM(amount) AS revenue
  FROM orders
  WHERE status = 'completed'
  GROUP BY date_format(order_date, 'yyyy-MM')
)
SELECT month, revenue,
       SUM(revenue) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue
FROM monthly
ORDER BY month
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("2023-01", 1020, 1020), ("2023-02", 655, 1675), ("2023-03", 1320, 2995)]

### Question 5

`LAG(order_date)` brings the previous date onto each row, and `DATEDIFF(later, earlier)` counts the days between
them. 5 January to 10 February is 36 days. 10 February to 15 March is 33 days, because February 2023 has 28 days.
The first order has no previous date, so the difference is `NULL`.

In [ ]:
q = """
SELECT order_id, order_date,
       DATEDIFF(order_date, LAG(order_date) OVER (ORDER BY order_date, order_id)) AS days_since_previous
FROM orders
WHERE customer_id = 1
ORDER BY order_id
"""
spark.sql(q).show()
assert [(r.order_id, r.days_since_previous) for r in spark.sql(q).collect()] == [(1, None), (2, 36), (3, 33)]